# Proyecto BATSÚ — Notebook 04: Juez RLAIF

**Universidad Cenfotec — MISIA** · Alvaro Granados, Jose Leonel Hernández

Este notebook implementa el instrumento 3.6.3, la **Rúbrica Estructurada de Evaluación del Juez**, con el modelo fijado en la Sección 5.5: **Llama-3-8B-Instruct** (familia distinta al generador, decisión D4), servido con vLLM en la GPU de Colab. Para cada turno del corpus, el juez lee el contexto acumulado de la conversación y el mensaje del agente en ese turno, y califica de 1 a 5 las dos dimensiones que la literatura de validez de criterio identifica como predictoras de conversión (Chen et al., 2026; Sección 2.4):

- **Adquisición de necesidades** (*need elicitation*): si el mensaje ayuda a descubrir las necesidades, restricciones o preferencias del turista.
- **Estrategia de ritmo** (*pacing*): si la acción llega en el momento adecuado, sin presionar el cierre ni saturar de información.

El puntaje se normaliza a [−1, 1] (Tabla 5.9) y se escribe en la columna `r_juez` del corpus mediante un UPDATE por `id_transicion`, sin regenerar nada (decisión D6). La evaluación es determinista (temperatura 0) para que sea reproducible.

**Modo piloto.** Por defecto juzga solo los episodios que ya tienen texto en `textos_<etiqueta>.jsonl` dentro del alcance de `N_SEMILLAS`/`N_CORRIDAS`. Guardado incremental por turno; reanudable.

**Secciones**
1. Configuración: Drive, GPU, vLLM y acceso a Hugging Face
2. Parámetros
3. Carga del corpus y de los textos; construcción de las unidades de evaluación
4. Servidor del modelo juez
5. Rúbrica y prompt del juez
6. Evaluación por lotes con guardado incremental
7. Normalización, UPDATE del corpus y verificación


## 1. Configuración

Igual que en el notebook 03b: sesión con **GPU L4**, vLLM instalado (con reinicio de sesión la primera vez) y el secreto `HF_TOKEN`. Llama 3 exige haber aceptado la licencia de Meta en la página del modelo en Hugging Face (huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct); sin eso la descarga falla con error 401/403.

Se verifica la GPU después de cada reinicio: Colab puede reasignar una T4 sin avisar, y el juez en 16 bits necesita los 24 GB de la L4.


In [1]:
from google.colab import drive, userdata
drive.mount('/content/drive')

from pathlib import Path
import sys, json, time, re, os, subprocess
import numpy as np
import pandas as pd

NOMBRE_CARPETA = 'DatosBatsuv2'
candidatos = [p for p in Path('/content/drive/MyDrive').rglob(NOMBRE_CARPETA) if p.is_dir()]
assert candidatos, f'No se encontró la carpeta "{NOMBRE_CARPETA}"'
BASE_DIR = candidatos[0]
CORPUS_DIR = BASE_DIR / 'corpus'
sys.path.insert(0, str(BASE_DIR))

gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
print('GPU:', gpu)
assert 'L4' in gpu or 'A100' in gpu, 'Se requiere GPU L4 (o A100): Entorno de ejecución → Cambiar tipo de entorno de ejecución'


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: NVIDIA L4, 23034 MiB


In [2]:
import importlib
if importlib.util.find_spec('vllm') is None:
    print('Instalando vLLM (varios minutos)...')
    !pip install -q vllm 2>&1 | tail -1
    !pip uninstall -y -q torchaudio
    !pip install -q --no-deps torchvision --index-url https://download.pytorch.org/whl/cu130 2>&1 | tail -1
    print('\n>>> Instalación terminada. REINICIAR la sesión (Entorno de ejecución → Reiniciar sesión) y volver a correr desde la primera celda.')
else:
    import vllm, torch, torchvision
    print('vLLM', vllm.__version__, '· torch', torch.__version__, '· torchvision', torchvision.__version__)

!pip install -q openai 2>&1 | tail -1
from huggingface_hub import login
HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)
print('Sesión de Hugging Face iniciada')


vLLM 0.31.0 · torch 2.13.0+cu130 · torchvision 0.28.0+cu130
Sesión de Hugging Face iniciada


## 2. Parámetros

- **`ETIQUETA_CORPUS`**: corpus a juzgar (el mismo del notebook 02 y 03b).
- **`N_SEMILLAS`, `N_CORRIDAS`**: alcance del piloto; `None, None` juzga todo lo que tenga texto.
- **`MODELO_JUEZ`**: identificador exacto (decisión D4). Se registra en el manifiesto.
- **`PESO_NECESIDADES`, `PESO_RITMO`**: ponderación de las dos dimensiones al combinarlas. Chen et al. (2026) muestran que ambas predicen conversión con correlaciones similares (ρ = 0.37 y 0.35), por lo que se parte de pesos iguales; quedan como parámetro para la calibración.
- **`TEMPERATURA = 0`**: evaluación determinista.


In [3]:
ETIQUETA_CORPUS = 'v0_escala05_prior40'
N_SEMILLAS = None            # None = todas
N_CORRIDAS = None           # None = todas
MODELO_JUEZ = 'meta-llama/Meta-Llama-3-8B-Instruct'
PESO_NECESIDADES, PESO_RITMO = 0.5, 0.5
TEMPERATURA = 0.0
LOTE = 256                # turnos que se envían juntos al servidor (cada cuánto se guarda)

RUTA_CORPUS = CORPUS_DIR / f'corpus_{ETIQUETA_CORPUS}.parquet'
RUTA_TEXTOS = CORPUS_DIR / f'textos_{ETIQUETA_CORPUS}.jsonl'
RUTA_JUEZ = CORPUS_DIR / f'juez_{ETIQUETA_CORPUS}.jsonl'          # una línea por turno juzgado, incremental
assert RUTA_CORPUS.exists() and RUTA_TEXTOS.exists(), 'Faltan el corpus (notebook 02) o los textos (notebook 03b)'
print('Corpus:', RUTA_CORPUS.name, '· textos:', RUTA_TEXTOS.name, '· juez:', RUTA_JUEZ.name, '· modelo:', MODELO_JUEZ)


Corpus: corpus_v0_escala05_prior40.parquet · textos: textos_v0_escala05_prior40.jsonl · juez: juez_v0_escala05_prior40.jsonl · modelo: meta-llama/Meta-Llama-3-8B-Instruct


## 3. Carga del corpus y de los textos; unidades de evaluación

Se carga el corpus estructurado y los textos válidos, y se construye una **unidad de evaluación por turno**: el diálogo acumulado hasta el turno anterior (mensajes del agente y del turista), más el mensaje del agente en el turno actual, que es la respuesta que se califica. La respuesta del turista en ese mismo turno **no** se muestra al juez: es la reacción que la tabla de reacción ya decidió, y mostrarla filtraría el desenlace en la calificación. El juez evalúa la calidad de la acción del agente dada la conversación previa, que es lo que la Tabla 5.14 define como entrada.

Cada unidad conserva el `id_transicion` del corpus, que es la llave del UPDATE posterior. Las unidades ya juzgadas (presentes en el archivo incremental) se omiten.


In [9]:
import batsu_sim as bs
corpus = pd.read_parquet(RUTA_CORPUS)
semillas = bs.semillas.set_index('id')

textos = {}
for linea in open(RUTA_TEXTOS, encoding='utf-8'):
    d = json.loads(linea)
    if d['valido']:
        textos[(d['id_semilla'], d['corrida'])] = d['turnos']

claves = sorted(textos.keys())
ids_sel = sorted({k[0] for k in claves})[:N_SEMILLAS] if N_SEMILLAS else sorted({k[0] for k in claves})
claves = [k for k in claves if k[0] in ids_sel and (N_CORRIDAS is None or k[1] < N_CORRIDAS)]

hechos = set()
if RUTA_JUEZ.exists():
    for linea in open(RUTA_JUEZ, encoding='utf-8'):
        d = json.loads(linea)
        if d['valido']:
            hechos.add(d['id_transicion'])

def unidades_de(k):
    s = semillas.loc[k[0]]
    tr = corpus[(corpus.id_semilla == k[0]) & (corpus.corrida == k[1])].sort_values('turno')
    turnos_txt = textos[k]
    out = []
    for i, row in enumerate(tr.itertuples()):
        historial = [(t.get('agente', ''), t.get('turista', '')) for t in turnos_txt[:i]]
        out.append({'id_transicion': row.id_transicion, 'id_semilla': k[0], 'corrida': k[1], 'turno': row.turno,
                    'accion_nombre': row.accion_nombre, 'estado_antes': row.estado_antes, 'etapa_antes': row.etapa_antes,
                    'segmento': s['segmento'], 'tipo_turismo': s['tipo_turismo'], 'destino': s['destino'],
                    'presupuesto': s['presupuesto'], 'origen': s['origen'], 'km': round(float(s['distancia_haversine_km'])),
                    'activos': ', '.join(s['activos_lista']), 'activo_enviado': row.activo_enviado,
                    'historial': historial, 'mensaje_agente': turnos_txt[i].get('agente', '')})
    return out

unidades = [u for k in claves for u in unidades_de(k)]
pendientes = [u for u in unidades if u['id_transicion'] not in hechos]
print(f'Episodios con texto en alcance: {len(claves)} · turnos: {len(unidades)} · ya juzgados: {len(unidades)-len(pendientes)} · pendientes: {len(pendientes)}')


Episodios con texto en alcance: 6000 · turnos: 26343 · ya juzgados: 26197 · pendientes: 146


## 4. Servidor del modelo juez

Se levanta vLLM como servidor en segundo plano con Llama-3-8B-Instruct en 16 bits (~16 GB), igual que en el 03b. Los prompts del juez son cortos (contexto de pocos turnos más la rúbrica) y la salida es un JSON de dos números, así que `--max-model-len 2048` basta y deja más memoria para procesar muchos turnos a la vez.


In [10]:
import requests

LOG = '/content/vllm_juez.log'
if 'vllm_proc' not in globals() or vllm_proc.poll() is not None:
    vllm_proc = subprocess.Popen(
        ['vllm', 'serve', MODELO_JUEZ, '--dtype', 'half', '--max-model-len', '2048',
         '--gpu-memory-utilization', '0.90', '--enforce-eager', '--port', '8001', '--served-model-name', 'juez'],
        stdout=open(LOG, 'w'), stderr=subprocess.STDOUT, env={**os.environ, 'HF_TOKEN': HF_TOKEN})

t0 = time.time()
while True:
    try:
        if requests.get('http://localhost:8001/health', timeout=2).status_code == 200:
            break
    except requests.exceptions.ConnectionError:
        pass
    if vllm_proc.poll() is not None:
        print(open(LOG).read()[-4000:])
        raise RuntimeError('El servidor vLLM se detuvo; ver el final del log arriba')
    time.sleep(10)
print(f'Servidor del juez listo en {time.time()-t0:.0f} s')


Servidor del juez listo en 0 s


## 5. Rúbrica y prompt del juez

La rúbrica operacionaliza las dos dimensiones del instrumento 3.6.3 con anclas por nivel, para que el juez califique de forma consistente y no por impresión general:

| Nivel | Adquisición de necesidades | Estrategia de ritmo |
|---|---|---|
| 1 | Ignora lo que el turista necesita o dijo; no pregunta nada útil | Presiona el cierre o satura de información sin que el turista esté listo |
| 2 | Pregunta algo genérico o repite lo ya sabido | Actúa a destiempo (ofrece antes de entender, o demora sin razón) |
| 3 | Atiende parcialmente una necesidad o pide un dato relevante | Momento aceptable, sin exceso ni retraso claros |
| 4 | Identifica y atiende una necesidad concreta del turista | Acción bien sincronizada con el estado y la etapa de la conversación |
| 5 | Descubre o resuelve la necesidad clave con precisión y aporta valor | Momento exacto: la acción es justo lo que la conversación pedía |

El prompt entrega al juez el contexto (segmento, destino, estado afectivo detectado, etapa del embudo), el historial de mensajes previos y el mensaje del agente a calificar, y exige salida únicamente en JSON con las dos calificaciones y una justificación de una línea (útil para auditar, no para la recompensa).


In [11]:
RUBRICA = """RÚBRICA (califique de 1 a 5 cada dimensión)
Adquisición de necesidades: 1 = ignora lo que el turista necesita o dijo; 2 = pregunta algo genérico o repite lo ya sabido; 3 = atiende parcialmente una necesidad o pide un dato relevante; 4 = identifica y atiende una necesidad concreta; 5 = descubre o resuelve la necesidad clave con precisión y aporta valor.
Estrategia de ritmo: 1 = presiona el cierre o satura de información sin que el turista esté listo; 2 = actúa a destiempo; 3 = momento aceptable; 4 = acción bien sincronizada con el estado y la etapa de la conversación; 5 = momento exacto, justo lo que la conversación pedía."""

def construir_prompt_juez(u):
    hist = '\n'.join(f'MiPyme: {a}\nTurista: {t}' for a, t in u['historial']) or '(inicio de la conversación)'
    return f"""Actúe como evaluador experto de conversaciones comerciales de WhatsApp entre turistas y MiPymes turísticas de Costa Rica.
Su tarea es calificar ÚNICAMENTE el último mensaje de la MiPyme, dado el contexto y la conversación previa.

CONTEXTO
- Turista {u['segmento']}, interesado en {u['tipo_turismo']} en {u['destino']}; viaja desde {u['origen']} (unos {u['km']} km).
- Presupuesto declarado por el turista: {u['presupuesto']}. Activos que la MiPyme tiene disponibles: {u['activos']}.
- Estado afectivo detectado en el turista: {u['estado_antes']}. Etapa del embudo de ventas: {u['etapa_antes']}.

CONVERSACIÓN PREVIA
{hist}

MENSAJE DE LA MIPYME A CALIFICAR{f" (adjunta: {u['activo_enviado']})" if u['activo_enviado'] else ""}
{u['mensaje_agente']}

{RUBRICA}

Responda ÚNICAMENTE con JSON válido, sin texto antes ni después:
{{"adquisicion_necesidades": <1-5>, "ritmo": <1-5>, "justificacion": "<máximo 20 palabras>"}}"""

print(construir_prompt_juez(pendientes[0]) if pendientes else construir_prompt_juez(unidades[0]))


Actúe como evaluador experto de conversaciones comerciales de WhatsApp entre turistas y MiPymes turísticas de Costa Rica.
Su tarea es calificar ÚNICAMENTE el último mensaje de la MiPyme, dado el contexto y la conversación previa.

CONTEXTO
- Turista internacional, interesado en playa en Manuel Antonio; viaja desde Aeropuerto Juan Santamaria (unos 67 km).
- Presupuesto declarado por el turista: bajo. Activos que la MiPyme tiene disponibles: texto, imagen.
- Estado afectivo detectado en el turista: indeciso. Etapa del embudo de ventas: deseo.

CONVERSACIÓN PREVIA
MiPyme: Hola! 🌴 Estoy aquí para ayudarte con tu viaje a Costa Rica. Tenemos un paquete de precio medio que incluye transporte y alojamiento en Manuel Antonio. ¿Te interesa?
Turista: No sé si es lo mejor para mi presupuesto...

MENSAJE DE LA MIPYME A CALIFICAR
Entendido. Te recomendaremos a nuestro asesor humano para que te asesore mejor. 💼

RÚBRICA (califique de 1 a 5 cada dimensión)
Adquisición de necesidades: 1 = ignora lo que

## 6. Evaluación por lotes con guardado incremental

Cada lote de turnos se envía en paralelo al servidor; la respuesta se parsea, se valida que las dos calificaciones sean enteros entre 1 y 5, y se guarda una línea por turno en `juez_<etiqueta>.jsonl` con las calificaciones, la justificación y el modelo. Si la salida no es válida se reintenta una vez (con temperatura 0 el reintento es idéntico, así que se usa temperatura 0.3 solo en el reintento); si vuelve a fallar se guarda como inválida y el turno queda sin puntaje, lo que se reporta en la Sección 7.


In [12]:
from openai import OpenAI
from concurrent.futures import ThreadPoolExecutor

cliente = OpenAI(base_url='http://localhost:8001/v1', api_key='no-key')

def extraer_json(texto):
    m = re.search(r'\{.*?\}', texto, re.S)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except json.JSONDecodeError:
        return None

ESQUEMA_JUEZ = {'type': 'object',
                'properties': {'adquisicion_necesidades': {'type': 'integer', 'minimum': 1, 'maximum': 5},
                               'ritmo': {'type': 'integer', 'minimum': 1, 'maximum': 5},
                               'justificacion': {'type': 'string', 'maxLength': 200}},
                'required': ['adquisicion_necesidades', 'ritmo', 'justificacion'], 'additionalProperties': False}

def _uno(prompt, temperatura):
    r = cliente.chat.completions.create(
        model='juez', messages=[{'role': 'user', 'content': prompt}],
        temperature=temperatura, max_tokens=200,
        response_format={'type': 'json_schema', 'json_schema': {'name': 'rubrica', 'schema': ESQUEMA_JUEZ}})
    return r.choices[0].message.content

def juzgar(lote, intento=1):
    temp = TEMPERATURA if intento == 1 else 0.3
    prompts = [construir_prompt_juez(u) for u in lote]
    with ThreadPoolExecutor(max_workers=min(64, len(prompts))) as ex:
        respuestas = list(ex.map(lambda p: _uno(p, temp), prompts))
    resultados, fallidos = [], []
    for u, resp in zip(lote, respuestas):
        d = extraer_json(resp)
        try:
            an, ri = int(d['adquisicion_necesidades']), int(d['ritmo'])
            ok = 1 <= an <= 5 and 1 <= ri <= 5
        except (TypeError, KeyError, ValueError):
            ok = False
        if ok:
            resultados.append({'id_transicion': u['id_transicion'], 'valido': True, 'adquisicion_necesidades': an, 'ritmo': ri,
                               'justificacion': str(d.get('justificacion', ''))[:300], 'modelo': MODELO_JUEZ, 'intento': intento})
        elif intento == 1:
            fallidos.append(u)
        else:
            resultados.append({'id_transicion': u['id_transicion'], 'valido': False, 'crudo': resp[:500], 'modelo': MODELO_JUEZ, 'intento': intento})
    if fallidos:
        resultados += juzgar(fallidos, intento=2)
    return resultados

total = len(pendientes); hechos_ahora = 0; t0 = time.time()
n_turnos_corpus = len(corpus)
print(f'Pendientes: {total}')
with open(RUTA_JUEZ, 'a', encoding='utf-8') as f:
    for i in range(0, total, LOTE):
        lote = pendientes[i:i + LOTE]
        for r in juzgar(lote):
            f.write(json.dumps(r, ensure_ascii=False) + '\n')
        f.flush()
        hechos_ahora += len(lote)
        seg = time.time() - t0
        por_turno = seg / hechos_ahora
        print(f'{hechos_ahora}/{total} turnos · {seg/60:.1f} min · {por_turno:.2f} s/turno · '
              f'proyección corpus completo ({n_turnos_corpus:,} turnos): {por_turno*n_turnos_corpus/3600:.2f} h')
print('Evaluación terminada' if total else 'No había turnos pendientes')


Pendientes: 146
146/146 turnos · 0.7 min · 0.29 s/turno · proyección corpus completo (26,343 turnos): 2.09 h
Evaluación terminada


## 7. Normalización, UPDATE del corpus y verificación

La calificación combinada es el promedio ponderado de las dos dimensiones (1 a 5), normalizado a [−1, 1] como pide la Tabla 5.9: `r_juez = (puntaje − 3) / 2`. Se escribe en la columna `r_juez` del corpus por `id_transicion` y se recalcula `reward = r_shaping + r_trust + r_juez + r_terminal`. Antes de sobrescribir el Parquet se guarda una copia sin juez (`corpus_<etiqueta>_sinjuez.parquet`), de modo que la iteración de D6 quede documentada.

Se verifican: proporción de turnos juzgados válidos, distribución de las dos calificaciones, calificación promedio por acción del agente (debe ser coherente: ofrecer paquete alto a un ansioso debería puntuar bajo en ritmo), y, como anticipo de la técnica 3.7.2, la correlación de Spearman entre el puntaje del juez y la recompensa terminal del episodio.


In [16]:
from scipy.stats import spearmanr
import hashlib, inspect

juez = pd.DataFrame([json.loads(l) for l in open(RUTA_JUEZ, encoding='utf-8')])
validos = juez[juez['valido']].drop_duplicates('id_transicion', keep='last')
sin_puntaje = len(set(juez['id_transicion']) - set(validos['id_transicion']))
print(f'Registros en el archivo: {len(juez)} · turnos con calificación válida: {len(validos)} · turnos sin calificación: {sin_puntaje}')
print('Adquisición de necesidades:', validos['adquisicion_necesidades'].value_counts().sort_index().to_dict())
print('Ritmo:', validos['ritmo'].value_counts().sort_index().to_dict())

validos = validos.assign(puntaje=PESO_NECESIDADES * validos['adquisicion_necesidades'] + PESO_RITMO * validos['ritmo'])
validos = validos.assign(r_juez=(validos['puntaje'] - 3.0) / 2.0)

# UPDATE del corpus (D6): copia sin juez, luego columna r_juez y reward recalculado
ruta_sinjuez = CORPUS_DIR / f'corpus_{ETIQUETA_CORPUS}_sinjuez.parquet'
if not ruta_sinjuez.exists():
    corpus.assign(r_juez=0.0).to_parquet(ruta_sinjuez, index=False)
mapa = validos.set_index('id_transicion')['r_juez']
corpus['r_juez'] = corpus['id_transicion'].map(mapa).fillna(0.0)
corpus['reward'] = corpus['r_shaping'] + corpus['r_trust'] + corpus['r_juez'] + corpus['r_terminal']
corpus.to_parquet(RUTA_CORPUS, index=False)
cobertura = corpus['id_transicion'].isin(mapa.index).mean()
print(f'\nUPDATE aplicado: {len(mapa):,} transiciones con r_juez ({cobertura:.1%} del corpus); el resto queda en 0.0')

# Coherencia: puntaje por acción y por estado afectivo
c = corpus[corpus['id_transicion'].isin(mapa.index)]
print('\nPuntaje del juez (media, escala 1–5) por acción:')
print((c.groupby('accion_nombre')['r_juez'].mean() * 2 + 3).round(2).sort_values(ascending=False).to_string())
print('\nPuntaje del juez por estado afectivo antes del turno:')
print((c.groupby('estado_antes')['r_juez'].mean() * 2 + 3).round(2).to_string())

# Anticipo de 3.7.2: puntaje medio del juez por episodio vs recompensa terminal
ep = c.groupby(['id_semilla', 'corrida']).agg(juez_medio=('r_juez', 'mean'), terminal=('r_terminal', 'sum'))
if len(ep) > 5:
    rho, pval = spearmanr(ep['juez_medio'], ep['terminal'])
    print(f'\nSpearman juez medio del episodio vs recompensa terminal: ρ = {rho:.3f} (p = {pval:.3g}, n = {len(ep)})')

# Muestra de justificaciones
print('\nEjemplos:')
for _, r in validos.sample(min(3, len(validos)), random_state=0).iterrows():
    t = corpus.set_index('id_transicion').loc[r['id_transicion']]
    print(f'  [{r["id_transicion"]}] {t["accion_nombre"]} · {t["estado_antes"]} → AN={r["adquisicion_necesidades"]} R={r["ritmo"]} · {r["justificacion"]}')

# Manifiesto
huella_juez = hashlib.sha256((inspect.getsource(construir_prompt_juez) + RUBRICA).encode('utf-8')).hexdigest()[:16]
ruta_manif = CORPUS_DIR / f'manifiesto_{ETIQUETA_CORPUS}.json'
manif = json.load(open(ruta_manif, encoding='utf-8'))
manif['juez'] = {'completado': bool(cobertura > 0.999), 'modelo': MODELO_JUEZ, 'precision': 'fp16', 'motor': 'vllm ' + __import__('vllm').__version__,
                 'temperatura': TEMPERATURA, 'pesos': {'adquisicion_necesidades': PESO_NECESIDADES, 'ritmo': PESO_RITMO},
                 'normalizacion': 'r_juez = (puntaje - 3) / 2', 'fecha': time.strftime('%Y-%m-%d %H:%M:%S'),
                 'turnos_juzgados': int(len(mapa)), 'turnos_sin_puntaje': int(len(corpus) - len(mapa)),
                 'cobertura': round(float(cobertura), 4), 'prompt_sha256': huella_juez}
json.dump(manif, open(ruta_manif, 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
print('\nManifiesto actualizado:', ruta_manif.name, '· huella del prompt del juez:', huella_juez)


Registros en el archivo: 26489 · turnos con calificación válida: 26343 · turnos sin calificación: 0
Adquisición de necesidades: {1.0: 151, 2.0: 4222, 3.0: 10534, 4.0: 11361, 5.0: 75}
Ritmo: {1.0: 28, 2.0: 1010, 3.0: 5485, 4.0: 19663, 5.0: 157}

UPDATE aplicado: 26,343 transiciones con r_juez (100.0% del corpus); el resto queda en 0.0

Puntaje del juez (media, escala 1–5) por acción:
accion_nombre
ofrecer_paquete_medio    3.60
ofrecer_paquete_alto     3.57
ofrecer_paquete_bajo     3.57
solicitar_informacion    3.51
enviar_activo            3.35
cerrar_conversacion      3.14
escalar_humano           2.97

Puntaje del juez por estado afectivo antes del turno:
estado_antes
ansioso         3.46
desconfiado     3.23
entusiasmado    3.66
frustrado       3.29
indeciso        3.31

Spearman juez medio del episodio vs recompensa terminal: ρ = 0.204 (p = 2.33e-57, n = 6000)

Ejemplos:
  [CR-TUR-008-081-02] escalar_humano · indeciso → AN=2.0 R=2.0 · La respuesta no atiende directamente a la pregun

**Nota: segundo pase con decodificación guiada**

En el primer pase sobre los 26 343 turnos, 146 (0.55 %) quedaron sin calificación válida: la salida del juez no se pudo interpretar como JSON o traía valores fuera del rango 1–5, incluso tras el reintento. Para esos turnos se hizo un segundo pase con el mismo prompt (misma huella 1dcf58b9ba383117) y decodificación guiada por esquema JSON en vLLM, que restringe la forma de la salida sin alterar su contenido. Los 146 registros fallidos del primer pase se conservan en juez_<etiqueta>.jsonl como evidencia; la consolidación toma el último registro válido por turno. Cobertura final: 100 %. El mismo procedimiento se aplicó en el notebook 03b a 644 episodios del generador.